# Tutorial

## User

The user interacts only with the configuration file, not with Python code.

1) Open the configuration file, in YAML or JSON.
   
The user can open the file on Visual Studio Code or other text editor. For example:
```yaml
simulation:
  sdate: '2012-08-01 00:00:00'
  edate: '2012-08-04 23:00:00'
  latitude: 43.61
  longitude: 3.87
  # unit: meters
  # param_type: float
  elevation: 44.0
  tzone: Europe/Paris
  output_index: ''
  unit_scene_length: cm
  # param_type: boolean
  hydraulic_structure: true
  negligible_shoot_resistance: false
  energy_budget: true
planting:
  spacing_between_rows: 3.6
  spacing_on_row: 1
  row_angle_with_south: 140.0
  ```

simulation and planting are sections, sdate, edate, latitude and all belows to simulation and planting are parameters. And all the lines that start with # are comments.

2) Modify a value

The user can change any value, he can change it directly on the JSON or YML file. 

3) Add a new parameter in the section, the name has to be unique in the section.

```yaml
planting:
  spacing_between_rows: 3.6
  spacing_on_row: 1
  row_angle_with_south: 140.0
  new_parameter: 45
  ```

4) Add a new section, for example, I add numerical_resolution with their parameters:

```yaml
simulation:
  sdate: '2012-08-01 00:00:00'
  edate: '2012-08-04 23:00:00'
  latitude: 43.61
  longitude: 3.87
  # unit: meters
  # param_type: float
  elevation: 44.0
  tzone: Europe/Paris
  output_index: ''
  unit_scene_length: cm
  # param_type: boolean
  hydraulic_structure: true
  negligible_shoot_resistance: false
  energy_budget: true
planting:
  spacing_between_rows: 3.6
  spacing_on_row: 1
  row_angle_with_south: 140.0
numerical_resolution:
# param_type: integer
  max_iter: 100
# param_type: float
  psi_step: 1.0
# unit: seconds
  psi_error_threshold: 0.05
  t_step: 1.0
  t_error_threshold: 0.02
  ```

5) Add new comment, only in YAML, comment start with #.

6) Delete a parameter, section or comment.










## Modeler

The modeler constructs the configuration. A Config is a Python dictionary that stores sections and parameters. A Config is built from a list of ModelUnit objects.


First, we have 3 classes: Parameter, ModelUnit and Config. 

In [28]:
@dataclass 
class Parameter:
    '''The dataclass Parameter has two mandatory parameters name and value, the others parameters are optional to the config, if
    we add them this would be commented, if we don't add them, we don't have comment. We also convert this class in dictionary.'''
    name: str
    value: any = None
    description : any = None
    unit : any = None
    param_type : any = None
    uid : any = None
    uri : any = None


    def __to_dict__(self):
        return {self.name: self.value}
    
    def __str__(self):
        return (
            f"name={self.name}, value={self.value}"
        )

In [29]:
class ModelUnit(dict):
    '''ModelUnit is a dictionary with a section name and a list of parameters '''
    def __init__(self, name, parameters: list):
        super().__init__({name: {p.name: p.value for p in parameters}})
        self.name = name
        self.parameters = parameters

    def __str__(self):
        return f"name={self.name}, parameters={dict(self)}"

In [42]:
class Config(dict):
    """Configuration of OpenAlea models
    Config is a dictionary that contains a list of model units. With the config can:
    -Load a new JSON or YAML file.
    -Dump a JSON or YAML file.
    -Add new sections.
    -Add new comments.
    -The quantity of units.
    
    TODO : Documentation to write
    """

    def __init__(self, model_unit_configs: list):
        """Initialize the configuration with a list of unit configurations."""

        super().__init__()
        self.model_unit_configs = model_unit_configs
        self.params_comments = {}
        self.section_comments = {}

        for unit in model_unit_configs:
            self.update(unit)

    def add_section(self, unit):
        self.model_unit_configs.append(unit)
        self.update(unit)

    def dump(self, filename: str):
        """Dump configuration to a file."""

        extension = filename.split(".")[-1]

        if extension in ("yml", "yaml"):
            yaml = YAML()
            cm = to_commented_map(self)
            

            add_comment(cm, self.params_comments)
            add_comment2(cm, self.model_unit_configs)
            add_section_comments(cm, self.section_comments)


            with open(filename, "w") as f:
                yaml.dump(cm, f)

        elif extension == "json":
            _dump_json(dict(self), filename)


The first thing the Modeler do is generate new parameters with an instance of the Parameter class. And an instance of the ModelUnit class, ModelUnit is a dictionary that contains a list of Parameters. And the config contains a list of model units and can generate a new config. A Config is built from a list of ModelUnit objects.

In [46]:
p_sdate = Parameter("sdate", "2012-08-01 00:00:00", "Start date of the simulation")
p_edate = Parameter("edate", "2012-08-04 23:00:00", "End date of the simulation")
p_lat = Parameter("latitude", 43.61, None, "degrees", "float")
p_longitude = Parameter("longitude", 3.87, "Longitude of the simulation", "degrees", "float")
p_elevation = Parameter("elevation", 44.0, None, "meters", "float")
p_tzone = Parameter("tzone", "Europe/Paris", "Time zone", None, "string")
p_output_index = Parameter("output_index", "")
p_unit_scene_length = Parameter("unit_scene_length", "cm")
p_hydraulic_structure = Parameter("hydraulic_structure", True, None, None, "boolean")
p_negligible_shoot_resistance = Parameter("negligible_shoot_resistance", False)
p_energy_budget = Parameter("energy_budget", True)
parameters = [p_sdate, p_edate, p_lat, p_longitude, p_elevation, p_tzone, p_output_index, p_unit_scene_length, p_hydraulic_structure, p_negligible_shoot_resistance, p_energy_budget]
simulation = ModelUnit('simulation', parameters)
config = Config([simulation])
print(config)

{'simulation': {'sdate': '2012-08-01 00:00:00', 'edate': '2012-08-04 23:00:00', 'latitude': 43.61, 'longitude': 3.87, 'elevation': 44.0, 'tzone': 'Europe/Paris', 'output_index': '', 'unit_scene_length': 'cm', 'hydraulic_structure': True, 'negligible_shoot_resistance': False, 'energy_budget': True}}


The modeler can also load and dump a configuration file as config.dump("params.yml") or config.dump("params.json"). The modeler can also add new sections, for example *config.add_section(planting)*. In this example, I can add new section.

In [47]:
p_spacing_between_rows = Parameter("spacing_between_rows", 3.6, "Distance between planting rows", "meters", "float")
p_spacing_on_row = Parameter("spacing_on_row", 1, None, "meters", "float")
p_row_angle_with_south = Parameter("row_angle_with_south", 140.0, None, "degrees", "float")

parameters = [p_spacing_between_rows, p_spacing_on_row, p_row_angle_with_south]
planting = ModelUnit('planting', parameters)
config.add_section(planting)

## Developer

The developer receives a config object. Config unherite from a dict, the developer can access to the parameters value. He can initialize the model and call run().